# PyTestLab Compliance and Audit Tutorial

Sign a simulated measurement, record an audit entry, and check the signature with an
explicit trust anchor. The notebook uses an ephemeral local key and does not contact a
timestamping service. The key is for demonstration only.

## 1. Sign a simulated measurement

In [ ]:
import sqlite3
from contextlib import closing
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

from pytestlab import AutoInstrument
from pytestlab.compliance import audited, signed

scratch = TemporaryDirectory(prefix="pytestlab-compliance-")
artifact_dir = Path(scratch.name)
private_pem = artifact_dir / "demo-private.pem"
public_pem = artifact_dir / "demo-public.pem"
audit_db = artifact_dir / "audit.sqlite"
key = ec.generate_private_key(ec.SECP256R1())
private_pem.write_bytes(
    key.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.PKCS8,
        serialization.NoEncryption(),
    )
)
public_pem.write_bytes(
    key.public_key().public_bytes(
        serialization.Encoding.PEM,
        serialization.PublicFormat.SubjectPublicKeyInfo,
    )
)

@audited(audit_db=audit_db)
@signed(key_file=private_pem)
def measure_voltage():
    dmm = AutoInstrument.from_preset("keysight/34470A", simulate=True)
    try:
        reading = dmm.measure("dc_voltage", range_val=10, uncertainty=False)
        return {"voltage_v": float(reading)}
    finally:
        dmm.close()

result = measure_voltage()
print(result.data)
print(f"Signature algorithm: {result.signature.algorithm}")

## 2. Inspect the local audit trail

In [ ]:
with closing(sqlite3.connect(audit_db)) as connection:
    entries = connection.execute(
        "SELECT event_type, success, result_hash FROM audit_log ORDER BY id"
    ).fetchall()

assert entries == [("MEASUREMENT", 1, result.audit_record.result_hash)]
print(f"Recorded {entries[0][0]} with result hash {entries[0][2]}")

## 3. Verify the original and detect a modified result

In [ ]:
original = result.verify(trust_anchor=public_pem)
modified = replace(result, data={"voltage_v": result.data["voltage_v"] + 1.0})
tampered = modified.verify(trust_anchor=public_pem)

assert original.signature_valid is True
assert tampered.signature_valid is False
print("Original signature verified; the modified reading was rejected.")

scratch.cleanup()

## Conclusion

The signature covers the measurement data; the SQLite entry records its execution.
The public key used here is a local demonstration trust anchor, not a production
identity or independent attestation of the database.